<a href="https://colab.research.google.com/github/dzidz1/Freeuni_ML_Walmart_Sales_Forecasting/blob/main/tree_based_models/model_experiment_XGBoost.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1. Setup & MLflow

In [1]:
!pip install -q -U xgboost mlflow dagshub

from google.colab import userdata
import dagshub

token = userdata.get("DAGSHUB_TOKEN")
assert token, "DAGSHUB_TOKEN secret missing or notebook access not enabled"

dagshub.auth.add_app_token(token)
dagshub.init(repo_owner="adzid23", repo_name="Freeuni_ML_Walmart_Sales_Forecasting", mlflow=True)

import mlflow
mlflow.set_experiment("XGBoost_Training")

import xgboost as xgb
print("xgboost", xgb.__version__)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 kB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 112.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 106.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 90.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.1/273.1 kB 28.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 264.7/264.7 kB 26.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 135.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.8/148.8 kB 16.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2

Accessing as adzid23

Initialized MLflow to track repo "adzid23/Freeuni_ML_Walmart_Sales_Forecasting"

Repository adzid23/Freeuni_ML_Walmart_Sales_Forecasting initialized!

2026/07/12 16:12:06 INFO mlflow.tracking.fluent: Experiment with name 'XGBoost_Training' does not exist. Creating a new experiment.


xgboost 3.3.0


## 2. Data

In [2]:
import os, zipfile
os.environ["KAGGLE_API_TOKEN"] = userdata.get("KAGGLE_KEY")

!kaggle competitions download -c walmart-recruiting-store-sales-forecasting -q

os.makedirs("walmart_data", exist_ok=True)
with zipfile.ZipFile("walmart-recruiting-store-sales-forecasting.zip") as z:
    z.extractall("walmart_data")
for f in os.listdir("walmart_data"):
    if f.endswith(".zip"):
        with zipfile.ZipFile(f"walmart_data/{f}") as z:
            z.extractall("walmart_data")

import numpy as np, pandas as pd
train  = pd.read_csv("walmart_data/train.csv", parse_dates=["Date"])
test   = pd.read_csv("walmart_data/test.csv",  parse_dates=["Date"])
stores = pd.read_csv("walmart_data/stores.csv")
feats  = pd.read_csv("walmart_data/features.csv", parse_dates=["Date"])
print(train.shape, test.shape, stores.shape, feats.shape)
print("Train:", train["Date"].min().date(), "→", train["Date"].max().date())

(421570, 5) (115064, 4) (45, 3) (8190, 12)
Train: 2010-02-05 → 2012-10-26


## 3. Exogenous covariates

In [3]:
md = [f"MarkDown{i}" for i in range(1, 6)]
feats[md] = feats[md].fillna(0.0)
feats["Total_MarkDown"] = feats[md].sum(axis=1)
feats["Num_MarkDowns"]  = (feats[md] > 0).sum(axis=1)
feats["Max_MarkDown"]   = feats[md].max(axis=1)
feats = feats.sort_values(["Store", "Date"])
feats[["CPI", "Unemployment"]] = feats.groupby("Store")[["CPI", "Unemployment"]].ffill()
exog = feats.drop(columns=["IsHoliday"] + md)

print(exog.columns.tolist())
print("NaNs left:", exog.isna().sum().sum())

['Store', 'Date', 'Temperature', 'Fuel_Price', 'CPI', 'Unemployment', 'Total_MarkDown', 'Num_MarkDowns', 'Max_MarkDown']
NaNs left: 0


## 4. Holiday calendar

In [4]:
HOLIDAYS = {k: pd.to_datetime(v) for k, v in {
    "superbowl":    ["2010-02-12", "2011-02-11", "2012-02-10", "2013-02-08"],
    "laborday":     ["2010-09-10", "2011-09-09", "2012-09-07", "2013-09-06"],
    "thanksgiving": ["2010-11-26", "2011-11-25", "2012-11-23", "2013-11-29"],
    "christmas":    ["2010-12-31", "2011-12-30", "2012-12-28", "2013-12-27"],
}.items()}
ALL_HOLIDAYS = pd.DatetimeIndex(sorted(set().union(*HOLIDAYS.values())))

def wks_from(dates, holiday_dates):
    """Signed weeks from nearest occurrence: negative = before the holiday."""
    d = np.asarray(dates.values[:, None] - holiday_dates.values[None, :],
                   dtype="timedelta64[D]").astype(float)
    return d[np.arange(len(d)), np.abs(d).argmin(axis=1)] / 7.0

chk = pd.Series(wks_from(pd.Series(pd.date_range("2012-12-07", periods=6, freq="W-FRI")),
                         HOLIDAYS["christmas"]),
                index=pd.date_range("2012-12-07", periods=6, freq="W-FRI"))
print(chk)

2012-12-07   -3.0
2012-12-14   -2.0
2012-12-21   -1.0
2012-12-28    0.0
2013-01-04    1.0
2013-01-11    2.0
Freq: W-FRI, dtype: float64


## 5. Panel builder - one leak-free featurizer

In [5]:
HORIZON, LAGS, ROLLS = 13, [13, 26, 39, 52], [13, 26]

def build_panel(history, target=None):
    hist = history[["Store", "Dept", "Date", "Weekly_Sales"]].copy()
    hist["is_target"] = False
    if target is not None and len(target):
        tgt = target[["Store", "Dept", "Date"]].copy()
        tgt["Weekly_Sales"] = np.nan
        tgt["is_target"] = True
        cutoff = tgt["Date"].min()
        df = pd.concat([hist, tgt], ignore_index=True)
    else:
        cutoff = hist["Date"].max() + pd.Timedelta(weeks=1)
        df = hist

    df = df.set_index(["Store", "Dept", "Date"]).sort_index()

    # full W-FRI grid per series
    bounds = df.reset_index().groupby(["Store", "Dept"])["Date"].agg(["min", "max"])
    idx = pd.MultiIndex.from_tuples(
        [(s, d, w) for (s, d), r in bounds.iterrows()
                   for w in pd.date_range(r["min"], r["max"], freq="W-FRI")],
        names=["Store", "Dept", "Date"])
    df = df.reindex(idx)
    df["is_target"] = df["is_target"].fillna(False).astype(bool)
    df["was_observed"] = df["Weekly_Sales"].notna() & ~df["is_target"]

    # zero-fill gap weeks (history only)
    dates = df.index.get_level_values("Date")
    gap = df["Weekly_Sales"].isna() & (dates < cutoff) & ~df["is_target"]
    df.loc[gap, "Weekly_Sales"] = 0.0

    # lags & rolling stats on a masked copy (gap zeros -> NaN)
    masked = df["Weekly_Sales"].where(~gap)
    g = masked.groupby(level=["Store", "Dept"])
    for k in LAGS:
        df[f"lag_{k}"] = g.shift(k)
    base = g.shift(HORIZON)
    gb = base.groupby(level=["Store", "Dept"])
    for w in ROLLS:
        df[f"rmean_{w}"] = gb.rolling(w, min_periods=4).mean().droplevel([0, 1])
        df[f"rstd_{w}"]  = gb.rolling(w, min_periods=4).std().droplevel([0, 1])

    # per-series aggregates from pre-cutoff observed data only
    agg = (masked[dates < cutoff]
           .groupby(level=["Store", "Dept"]).agg(["mean", "std", "median"]))
    agg.columns = ["series_mean", "series_std", "series_median"]
    df = df.join(agg).reset_index()

    # calendar
    df["week"]  = df["Date"].dt.isocalendar().week.astype(int)
    df["month"] = df["Date"].dt.month
    df["year"]  = df["Date"].dt.year
    df["week_sin"] = np.sin(2 * np.pi * df["week"] / 52)
    df["week_cos"] = np.cos(2 * np.pi * df["week"] / 52)

    # holidays: flag + signed weeks-to-nearest per holiday
    df["IsHoliday"] = df["Date"].isin(ALL_HOLIDAYS).astype(int)
    udates = pd.DatetimeIndex(sorted(df["Date"].unique()))
    for name, hd in HOLIDAYS.items():
        d = (udates.values[:, None] - hd.values[None, :]) / np.timedelta64(1, "W")
        nearest = d[np.arange(len(udates)), np.abs(d).argmin(axis=1)]
        df[f"wks_from_{name}"] = df["Date"].map(pd.Series(nearest, index=udates))

    # static + exogenous + ratios
    df = df.merge(stores, on="Store", how="left").merge(exog, on=["Store", "Date"], how="left")
    df["md_intensity"] = df["Total_MarkDown"] / df["Size"]
    df["cpi_unemp"]    = df["CPI"] / df["Unemployment"]
    df["fuel_cpi"]     = df["Fuel_Price"] / df["CPI"]
    for c in ["Store", "Dept", "Type"]:
        df[c] = df[c].astype("category")
    return df

## 6. Metric & the 600-series subset

In [6]:
def wmae(y, yhat, is_holiday):
    w = np.where(is_holiday, 5, 1)
    return np.sum(w * np.abs(np.asarray(y) - np.asarray(yhat))) / np.sum(w)

lengths = train.groupby(["Store", "Dept"])["Date"].nunique()
full_len = lengths[lengths == 143].index
means = (train.set_index(["Store", "Dept"]).loc[full_len]
              .groupby(["Store", "Dept"])["Weekly_Sales"].mean())
top300 = means.nlargest(300).index
rest = means.drop(top300)
rng = np.random.default_rng(7)
rand300 = rest.index[rng.choice(len(rest), 300, replace=False)]
val_series = set(top300) | set(rand300)

print(len(val_series), sorted(val_series)[:3])

600 [(1, 2), (1, 9), (1, 12)]


## 7. Train / validation matrices

In [7]:
val_start = pd.Timestamp("2012-08-03")
hist = train[train["Date"] < val_start]

val_dates = pd.date_range(val_start, train["Date"].max(), freq="W-FRI")   # 13 weeks
val_grid = (hist[["Store", "Dept"]].drop_duplicates()
            .merge(pd.DataFrame({"Date": val_dates}), how="cross"))

panel = build_panel(hist, val_grid)
train_df = panel[panel["was_observed"]].copy()
val_df   = panel[panel["is_target"]].copy()

actual = train.loc[train["Date"] >= val_start, ["Store", "Dept", "Date", "Weekly_Sales"]]
val_df = val_df.merge(actual, on=["Store", "Dept", "Date"], how="left", suffixes=("", "_true"))
val_df["y_true"] = val_df["Weekly_Sales_true"].fillna(0)
val_df["in600"] = pd.MultiIndex.from_frame(val_df[["Store", "Dept"]]).isin(val_series)

# leakage sanity checks
assert train_df["Date"].max() < val_start
assert (val_df["Date"] >= val_start).all()
assert val_df["in600"].sum() == 600 * 13
print("train:", train_df.shape, "| val:", val_df.shape, "| val in600:", val_df["in600"].sum())

/tmp/ipykernel_491/1761351721.py:25: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df["is_target"] = df["is_target"].fillna(False).astype(bool)


train: (383040, 39) | val: (43186, 42) | val in600: 7800


## 8. Feature set v3 + target

In [8]:
# v2/v3 features (locked LightGBM recipe), applied to both frames identically
sv = panel.set_index(["Store", "Dept", "Date"])["Weekly_Sales"]
base = sv.groupby(level=["Store", "Dept"], observed=True).shift(HORIZON)
r52 = (base.groupby(level=["Store", "Dept"], observed=True)
           .rolling(52, min_periods=13).mean().droplevel([0, 1]).rename("rmean_52"))
extra_lags = {k: sv.groupby(level=["Store", "Dept"], observed=True).shift(k).rename(f"lag_{k}")
              for k in [51, 53]}

obs = train[train["Date"] < val_start]
dept_med  = obs.groupby("Dept")["Weekly_Sales"].median().rename("dept_median")
store_med = obs.groupby("Store")["Weekly_Sales"].median().rename("store_median")

def add_v23(df):
    df = df.merge(r52.reset_index(), on=["Store", "Dept", "Date"], how="left")
    for k in [51, 53]:
        df = df.merge(extra_lags[k].reset_index(), on=["Store", "Dept", "Date"], how="left")
    denom = df["series_median"].replace(0, np.nan)
    for k in LAGS:
        df[f"lag_{k}_ratio"] = df[f"lag_{k}"] / denom
    df["rmean_13_ratio"] = df["rmean_13"] / denom
    df["lag_52_smooth"] = df[["lag_51", "lag_52", "lag_53"]].mean(axis=1)
    df["lag_52_smooth_ratio"] = df["lag_52_smooth"] / denom
    df["Dept_i"] = df["Dept"].astype(int); df["Store_i"] = df["Store"].astype(int)
    df = (df.merge(dept_med, left_on="Dept_i", right_index=True, how="left")
            .merge(store_med, left_on="Store_i", right_index=True, how="left")
            .drop(columns=["Dept_i", "Store_i"]))
    return df

train_df = add_v23(train_df)
val_df   = add_v23(val_df)

# explicit category alignment (the LightGBM notebook's crash, fixed up front)
CATS = {c: train_df[c].astype("category").cat.categories for c in ["Store", "Dept", "Type"]}
for df in (train_df, val_df):
    for c, cats in CATS.items():
        df[c] = pd.Categorical(df[c], categories=cats)

FEATURES = ([f"lag_{k}" for k in LAGS] +
            [f"rmean_{w}" for w in ROLLS] + [f"rstd_{w}" for w in ROLLS] +
            ["series_mean", "series_std", "series_median",
             "week", "month", "year", "week_sin", "week_cos", "IsHoliday"] +
            [f"wks_from_{h}" for h in HOLIDAYS] +
            ["Store", "Dept", "Type", "Size",
             "Temperature", "Fuel_Price", "CPI", "Unemployment",
             "Total_MarkDown", "Num_MarkDowns", "Max_MarkDown",
             "md_intensity", "cpi_unemp", "fuel_cpi"])
FEATURES_V3 = (FEATURES + [f"lag_{k}_ratio" for k in LAGS] +
               ["rmean_13_ratio", "rmean_52",
                "lag_52_smooth", "lag_52_smooth_ratio", "dept_median", "store_median"])

X_tr, X_val = train_df[FEATURES_V3], val_df[FEATURES_V3]
y_tr_raw, y_val_raw = train_df["Weekly_Sales"], val_df["y_true"]
shift = float(max(0.0, -y_tr_raw.min()))
y_tr_log, y_val_log = np.log1p(y_tr_raw + shift), np.log1p(y_val_raw + shift)
w_tr  = np.where(train_df["IsHoliday"] == 1, 5.0, 1.0)
w_val = np.where(val_df["IsHoliday"] == 1, 5.0, 1.0)

print(X_tr.shape, X_val.shape, "| shift:", round(shift, 2), "| n_features:", len(FEATURES_V3))

(383040, 45) (43186, 45) | shift: 4988.94 | n_features: 45


## 9. Port of the LightGBM best config

In [10]:
def wmae_orig_scale(y_true_log, y_pred_log, sample_weight=None):
    """Early-stopping metric: WMAE on the original scale."""
    pred = np.expm1(y_pred_log) - shift
    true = np.expm1(y_true_log) - shift
    w = sample_weight if sample_weight is not None else np.ones_like(true)
    return float(np.average(np.abs(true - pred), weights=w))

XGB_PORT = dict(
    device="cuda", tree_method="hist",
    objective="reg:absoluteerror",
    learning_rate=0.02,
    grow_policy="lossguide", max_leaves=511, max_depth=0,
    min_child_weight=60,
    enable_categorical=True, max_cat_threshold=128,
    n_estimators=6000, early_stopping_rounds=200,
    verbosity=0,
)

with mlflow.start_run(run_name="XGBoost_Port_LGBM_Best"):
    mlflow.set_tags({"stage": "baseline", "model_family": "XGBoost", "author": "adzid23"})
    mlflow.set_tag("mlflow.note.content",
        "Direct port of LightGBM best config (lr=.02, 511 leaves, mcs=60, log1p "
        "target, 5x holiday weights, v3 features, same 13-wk split). Early stop "
        "on original-scale WMAE. Bars: LGBM single 2766.0 / ensemble 2757.2.")

    m = xgb.XGBRegressor(**XGB_PORT, random_state=7, eval_metric=wmae_orig_scale)
    m.fit(X_tr, y_tr_log, sample_weight=w_tr,
          eval_set=[(X_val, y_val_log)], sample_weight_eval_set=[w_val],
          verbose=200)

    p = np.expm1(m.predict(X_val)) - shift
    m6 = val_df.assign(pred=p)[val_df["in600"]]
    w6 = wmae(m6["y_true"], m6["pred"], m6["IsHoliday"] == 1)
    wa = wmae(val_df["y_true"], p, val_df["IsHoliday"] == 1)

    mlflow.log_params({k: v for k, v in XGB_PORT.items()} |
                      {"seed": 7, "target": "log1p", "shift": shift,
                       "n_features": len(FEATURES_V3), "holiday_weight": 5})
    mlflow.log_metrics({"wmae_600": w6, "wmae_all": wa, "best_iteration": m.best_iteration})
    print(f"wmae_600: {w6:.1f} | wmae_all: {wa:.1f} | best_iter: {m.best_iteration}")

[0]	validation_0-mae:0.69654	validation_0-wmae_orig_scale:12534.83496
[200]	validation_0-mae:0.06667	validation_0-wmae_orig_scale:1665.96509
[400]	validation_0-mae:0.05585	validation_0-wmae_orig_scale:1237.75586
[600]	validation_0-mae:0.05512	validation_0-wmae_orig_scale:1225.26648
[800]	validation_0-mae:0.05491	validation_0-wmae_orig_scale:1222.21826
[1000]	validation_0-mae:0.05487	validation_0-wmae_orig_scale:1222.33386
[1043]	validation_0-mae:0.05488	validation_0-wmae_orig_scale:1222.87683
wmae_600: 2852.3 | wmae_all: 1221.4 | best_iter: 843
🏃 View run XGBoost_Port_LGBM_Best at: https://dagshub.com/adzid23/Freeuni_ML_Walmart_Sales_Forecasting.mlflow/#/experiments/9/runs/5aea48199d6c4a67bce735f02da7acdb
🧪 View experiment at: https://dagshub.com/adzid23/Freeuni_ML_Walmart_Sales_Forecasting.mlflow/#/experiments/9


## 10. Tuning

In [11]:
SWEEP = {
    "subsample08":      dict(subsample=0.8, colsample_bytree=0.8),
    "mcw20":            dict(min_child_weight=20),
    "mcw150":           dict(min_child_weight=150),
    "sub08_mcw20":      dict(subsample=0.8, colsample_bytree=0.8, min_child_weight=20),
    "leaves255_sub08":  dict(max_leaves=255, subsample=0.8, colsample_bytree=0.8),
}

results = {"port": 2852.3}
for name, override in SWEEP.items():
    cfg = {**XGB_PORT, **override}
    with mlflow.start_run(run_name=f"XGBoost_Tune_{name}"):
        mlflow.set_tags({"stage": "tuning", "model_family": "XGBoost", "author": "adzid23"})
        m = xgb.XGBRegressor(**cfg, random_state=7, eval_metric=wmae_orig_scale)
        m.fit(X_tr, y_tr_log, sample_weight=w_tr,
              eval_set=[(X_val, y_val_log)], sample_weight_eval_set=[w_val],
              verbose=False)
        p = np.expm1(m.predict(X_val)) - shift
        m6 = val_df.assign(pred=p)[val_df["in600"]]
        w6 = wmae(m6["y_true"], m6["pred"], m6["IsHoliday"] == 1)
        wa = wmae(val_df["y_true"], p, val_df["IsHoliday"] == 1)
        mlflow.log_params({**cfg, "seed": 7, "target": "log1p"})
        mlflow.log_metrics({"wmae_600": w6, "wmae_all": wa, "best_iteration": m.best_iteration})
        results[name] = w6
        print(f"{name:18s} wmae_600={w6:.1f} | wmae_all={wa:.1f} | it={m.best_iteration}")

print("\nbest:", min(results, key=results.get), min(results.values()))

subsample08        wmae_600=2863.4 | wmae_all=1221.8 | it=883
🏃 View run XGBoost_Tune_subsample08 at: https://dagshub.com/adzid23/Freeuni_ML_Walmart_Sales_Forecasting.mlflow/#/experiments/9/runs/d28f2ea444dd4df99e95e6d10e90e5e4
🧪 View experiment at: https://dagshub.com/adzid23/Freeuni_ML_Walmart_Sales_Forecasting.mlflow/#/experiments/9
mcw20              wmae_600=2914.7 | wmae_all=1230.9 | it=827
🏃 View run XGBoost_Tune_mcw20 at: https://dagshub.com/adzid23/Freeuni_ML_Walmart_Sales_Forecasting.mlflow/#/experiments/9/runs/cc0490faceb14df69a849e5051431cd4
🧪 View experiment at: https://dagshub.com/adzid23/Freeuni_ML_Walmart_Sales_Forecasting.mlflow/#/experiments/9
mcw150             wmae_600=2937.8 | wmae_all=1236.8 | it=1115
🏃 View run XGBoost_Tune_mcw150 at: https://dagshub.com/adzid23/Freeuni_ML_Walmart_Sales_Forecasting.mlflow/#/experiments/9/runs/cceaf08a4c1b478d9c21ae5a93391ce1
🧪 View experiment at: https://dagshub.com/adzid23/Freeuni_ML_Walmart_Sales_Forecasting.mlflow/#/experiment

## 11. Refinement

In [12]:
REFINE = {
    "leaves127_sub08": dict(max_leaves=127, subsample=0.8, colsample_bytree=0.8),
    "leaves63_sub08":  dict(max_leaves=63,  subsample=0.8, colsample_bytree=0.8),
    "leaves255_lr01":  dict(max_leaves=255, subsample=0.8, colsample_bytree=0.8,
                            learning_rate=0.01, n_estimators=12000),
}

for name, override in REFINE.items():
    cfg = {**XGB_PORT, **override}
    with mlflow.start_run(run_name=f"XGBoost_Tune_{name}"):
        mlflow.set_tags({"stage": "tuning", "model_family": "XGBoost", "author": "adzid23"})
        m = xgb.XGBRegressor(**cfg, random_state=7, eval_metric=wmae_orig_scale)
        m.fit(X_tr, y_tr_log, sample_weight=w_tr,
              eval_set=[(X_val, y_val_log)], sample_weight_eval_set=[w_val],
              verbose=False)
        p = np.expm1(m.predict(X_val)) - shift
        m6 = val_df.assign(pred=p)[val_df["in600"]]
        w6 = wmae(m6["y_true"], m6["pred"], m6["IsHoliday"] == 1)
        wa = wmae(val_df["y_true"], p, val_df["IsHoliday"] == 1)
        mlflow.log_params({**cfg, "seed": 7, "target": "log1p"})
        mlflow.log_metrics({"wmae_600": w6, "wmae_all": wa, "best_iteration": m.best_iteration})
        results[name] = w6
        print(f"{name:18s} wmae_600={w6:.1f} | wmae_all={wa:.1f} | it={m.best_iteration}")

print("\nbest so far:", min(results, key=results.get), round(min(results.values()), 1))

leaves127_sub08    wmae_600=2826.2 | wmae_all=1210.3 | it=2665
🏃 View run XGBoost_Tune_leaves127_sub08 at: https://dagshub.com/adzid23/Freeuni_ML_Walmart_Sales_Forecasting.mlflow/#/experiments/9/runs/ca20c246a6c84f00b132376beaf28f45
🧪 View experiment at: https://dagshub.com/adzid23/Freeuni_ML_Walmart_Sales_Forecasting.mlflow/#/experiments/9
leaves63_sub08     wmae_600=2832.8 | wmae_all=1212.7 | it=3701
🏃 View run XGBoost_Tune_leaves63_sub08 at: https://dagshub.com/adzid23/Freeuni_ML_Walmart_Sales_Forecasting.mlflow/#/experiments/9/runs/c7f0f3e8e70a46a8a2ff6848a32a4de3
🧪 View experiment at: https://dagshub.com/adzid23/Freeuni_ML_Walmart_Sales_Forecasting.mlflow/#/experiments/9
leaves255_lr01     wmae_600=2845.4 | wmae_all=1214.4 | it=2488
🏃 View run XGBoost_Tune_leaves255_lr01 at: https://dagshub.com/adzid23/Freeuni_ML_Walmart_Sales_Forecasting.mlflow/#/experiments/9/runs/26e2c2901a454e85bfa64ad728856f53
🧪 View experiment at: https://dagshub.com/adzid23/Freeuni_ML_Walmart_Sales_Forecast

## 12. Seed ensemble

In [13]:
BEST_XGB = {**XGB_PORT, "max_leaves": 255, "subsample": 0.8, "colsample_bytree": 0.8}
BEST_XGB.pop("early_stopping_rounds")
SEEDS, N_EST = [7, 77, 777], 1800

preds_log = []
with mlflow.start_run(run_name="XGBoost_SeedEnsemble"):
    mlflow.set_tags({"stage": "ensemble", "model_family": "XGBoost", "author": "adzid23"})
    mlflow.set_tag("mlflow.note.content",
        "3-seed average of best config (fixed 1800 iters, no per-seed early stop), "
        "log-space averaging. Single-seed best: 2823.2. LightGBM ensemble: 2757.2.")

    for sd in SEEDS:
        m = xgb.XGBRegressor(**{**BEST_XGB, "n_estimators": N_EST}, random_state=sd)
        m.fit(X_tr, y_tr_log, sample_weight=w_tr, verbose=False)
        preds_log.append(m.predict(X_val))
        p_i = np.expm1(preds_log[-1]) - shift
        m6 = val_df.assign(pred=p_i)[val_df["in600"]]
        print(f"seed {sd}: wmae_600={wmae(m6['y_true'], m6['pred'], m6['IsHoliday']==1):.1f}")

    p = np.expm1(np.mean(preds_log, axis=0)) - shift
    m6 = val_df.assign(pred=p)[val_df["in600"]]
    w6 = wmae(m6["y_true"], m6["pred"], m6["IsHoliday"] == 1)
    wa = wmae(val_df["y_true"], p, val_df["IsHoliday"] == 1)
    mlflow.log_params({**BEST_XGB, "seeds": str(SEEDS), "n_estimators_fixed": N_EST,
                       "target": "log1p", "shift": shift, "n_features": len(FEATURES_V3)})
    mlflow.log_metrics({"wmae_600": w6, "wmae_all": wa})
    print(f"\nensemble: wmae_600={w6:.1f} | wmae_all={wa:.1f} (single: 2823.2)")

seed 7: wmae_600=2823.2
seed 77: wmae_600=2861.8
seed 777: wmae_600=2850.3

ensemble: wmae_600=2831.0 | wmae_all=1210.0 (single: 2823.2)
🏃 View run XGBoost_SeedEnsemble at: https://dagshub.com/adzid23/Freeuni_ML_Walmart_Sales_Forecasting.mlflow/#/experiments/9/runs/00e6b3921292442e853c8b47641728d8
🧪 View experiment at: https://dagshub.com/adzid23/Freeuni_ML_Walmart_Sales_Forecasting.mlflow/#/experiments/9


In [14]:
def featurize(train_raw, target_raw, stores_raw, feats_raw):
    """Raw csv frames in -> (target-row features, history-row features), v3 set."""
    f = feats_raw.copy()
    md_ = [f"MarkDown{i}" for i in range(1, 6)]
    f[md_] = f[md_].fillna(0.0)
    f["Total_MarkDown"] = f[md_].sum(axis=1)
    f["Num_MarkDowns"]  = (f[md_] > 0).sum(axis=1)
    f["Max_MarkDown"]   = f[md_].max(axis=1)
    f = f.sort_values(["Store", "Date"])
    f[["CPI", "Unemployment"]] = f.groupby("Store")[["CPI", "Unemployment"]].ffill()
    ex = f.drop(columns=["IsHoliday"] + md_)

    global exog, stores
    _e, _s = exog, stores
    exog, stores = ex, stores_raw.copy()
    try:
        pan = build_panel(train_raw, target_raw)
    finally:
        exog, stores = _e, _s

    sv = pan.set_index(["Store", "Dept", "Date"])["Weekly_Sales"]
    base = sv.groupby(level=["Store", "Dept"], observed=True).shift(HORIZON)
    r52 = (base.groupby(level=["Store", "Dept"], observed=True)
               .rolling(52, min_periods=13).mean().droplevel([0, 1]).rename("rmean_52"))
    lag_extra = {k: sv.groupby(level=["Store", "Dept"], observed=True).shift(k).rename(f"lag_{k}")
                 for k in [51, 53]}
    dept_med  = train_raw.groupby("Dept")["Weekly_Sales"].median().rename("dept_median")
    store_med = train_raw.groupby("Store")["Weekly_Sales"].median().rename("store_median")

    def _v23(df):
        df = df.merge(r52.reset_index(), on=["Store", "Dept", "Date"], how="left")
        for k in [51, 53]:
            df = df.merge(lag_extra[k].reset_index(), on=["Store", "Dept", "Date"], how="left")
        denom = df["series_median"].replace(0, np.nan)
        for k in LAGS:
            df[f"lag_{k}_ratio"] = df[f"lag_{k}"] / denom
        df["rmean_13_ratio"] = df["rmean_13"] / denom
        df["lag_52_smooth"] = df[["lag_51", "lag_52", "lag_53"]].mean(axis=1)
        df["lag_52_smooth_ratio"] = df["lag_52_smooth"] / denom
        df["Dept_i"] = df["Dept"].astype(int); df["Store_i"] = df["Store"].astype(int)
        df = (df.merge(dept_med, left_on="Dept_i", right_index=True, how="left")
                .merge(store_med, left_on="Store_i", right_index=True, how="left")
                .drop(columns=["Dept_i", "Store_i"]))
        return df

    out = (_v23(pan[pan["is_target"]].copy())
           if target_raw is not None and len(target_raw) else pan.iloc[:0])
    hist = _v23(pan[pan["was_observed"]].copy())
    return out, hist

# --- refit on FULL train, 3 seeds, fixed iterations ---
_, tr_full = featurize(train, None, stores,
                       pd.read_csv("walmart_data/features.csv", parse_dates=["Date"]))

CATS = {c: tr_full[c].astype("category").cat.categories for c in ["Store", "Dept", "Type"]}
for c, cats in CATS.items():
    tr_full[c] = pd.Categorical(tr_full[c], categories=cats)

Xf = tr_full[FEATURES_V3]
yf_log = np.log1p(tr_full["Weekly_Sales"] + shift)
wf = np.where(tr_full["IsHoliday"] == 1, 5.0, 1.0)

final_models = []
for sd in SEEDS:
    m = xgb.XGBRegressor(**{**BEST_XGB, "n_estimators": N_EST}, random_state=sd)
    m.fit(Xf, yf_log, sample_weight=wf, verbose=False)
    m.set_params(device="cpu")          # shipped model must predict on CPU boxes
    final_models.append(m)
print("refit done:", len(final_models), "models on", Xf.shape)

/tmp/ipykernel_491/1761351721.py:25: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df["is_target"] = df["is_target"].fillna(False).astype(bool)


refit done: 3 models on (421570, 45)


## 14. Final pipeline (b) - pyfunc

In [15]:
import shutil
from mlflow.models import infer_signature

os.makedirs("pipeline_artifacts", exist_ok=True)
for f in ["train.csv", "features.csv", "stores.csv"]:
    shutil.copy(f"walmart_data/{f}", f"pipeline_artifacts/{f}")

class XGBWalmartPipeline(mlflow.pyfunc.PythonModel):
    def __init__(self, models, cats, features, shift):
        self.models, self.cats, self.features, self.shift = models, cats, features, shift

    def load_context(self, context):
        self.train_raw  = pd.read_csv(context.artifacts["train"],    parse_dates=["Date"])
        self.feats_raw  = pd.read_csv(context.artifacts["features"], parse_dates=["Date"])
        self.stores_raw = pd.read_csv(context.artifacts["stores"])

    def predict(self, context, model_input):
        tgt = model_input.copy()
        tgt["Date"] = pd.to_datetime(tgt["Date"])
        out, _ = featurize(self.train_raw, tgt[["Store", "Dept", "Date"]],
                           self.stores_raw, self.feats_raw)
        for c, cats in self.cats.items():
            out[c] = pd.Categorical(out[c], categories=cats)
        X = out[self.features]
        p_log = np.mean([m.predict(X) for m in self.models], axis=0)
        preds = pd.DataFrame({"Store": out["Store"].astype(int),
                              "Dept": out["Dept"].astype(int),
                              "Date": out["Date"],
                              "Weekly_Sales": np.expm1(p_log) - self.shift})
        return (tgt.merge(preds, on=["Store", "Dept", "Date"], how="left")
                   ["Weekly_Sales"].fillna(0.0))

pipeline = XGBWalmartPipeline(final_models, CATS, FEATURES_V3, shift)
sample_in = test[["Store", "Dept", "Date", "IsHoliday"]].head(5).reset_index(drop=True)

with mlflow.start_run(run_name="XGBoost_Final_Pipeline"):
    mlflow.set_tags({"stage": "pipeline", "model_family": "XGBoost", "author": "adzid23"})
    mlflow.set_tag("mlflow.note.content",
        "End-to-end pyfunc: raw test.csv rows in -> Weekly_Sales out. 3-seed "
        "XGBoost (255 leaves, sub/col 0.8, lr=.02, 1800 iters), log1p target, "
        "v3 features rebuilt internally from shipped csvs. Val (pre-refit): "
        "wmae_600=2831.0, wmae_all=1210.0. LightGBM ensemble: 2757.2.")
    mlflow.log_params({**BEST_XGB, "seeds": str(SEEDS), "n_estimators_fixed": N_EST,
                       "target": "log1p", "shift": shift,
                       "n_features": len(FEATURES_V3), "refit": "full_train"})
    mlflow.log_metrics({"wmae_600": 2831.0, "wmae_all": 1210.0})

    model_info = mlflow.pyfunc.log_model(
        name="xgboost_pipeline",
        python_model=pipeline,
        artifacts={"train":    "pipeline_artifacts/train.csv",
                   "features": "pipeline_artifacts/features.csv",
                   "stores":   "pipeline_artifacts/stores.csv"},
        signature=infer_signature(sample_in,
                                  pd.Series(np.zeros(len(sample_in)), name="Weekly_Sales")),
        pip_requirements=["xgboost", "pandas", "numpy", "scikit-learn"])

print("logged:", model_info.model_uri)

/usr/local/lib/python3.12/dist-packages/mlflow/pyfunc/utils/data_validation.py:187: UserWarning: Add type hints to the `predict` method to enable data validation and automatic signature inference during model logging. Check https://mlflow.org/docs/latest/model/python_model.html#type-hint-usage-in-pythonmodel for more details.
  color_warning(
/usr/local/lib/python3.12/dist-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/doc

🏃 View run XGBoost_Final_Pipeline at: https://dagshub.com/adzid23/Freeuni_ML_Walmart_Sales_Forecasting.mlflow/#/experiments/9/runs/68e102b57b60498bb6dde58303b2da31
🧪 View experiment at: https://dagshub.com/adzid23/Freeuni_ML_Walmart_Sales_Forecasting.mlflow/#/experiments/9
logged: models:/m-443a01e2c8574d4d98d11cfe2fd40eca


## 15. Round-trip + submission

In [16]:
loaded = mlflow.pyfunc.load_model(model_info.model_uri)

test_in = test[["Store", "Dept", "Date", "IsHoliday"]].reset_index(drop=True)
preds = loaded.predict(test_in)

smoke = test_in.assign(pred=preds.values)
smoke["Date"] = pd.to_datetime(smoke["Date"])
print("rows:", len(smoke), "| NaNs:", smoke["pred"].isna().sum())

# Christmas spike check: dept-1 store-1 around week 51 of 2012
s1d1 = smoke[(smoke["Store"] == 1) & (smoke["Dept"] == 1) &
             (smoke["Date"].between("2012-11-30", "2012-12-28"))]
print(s1d1[["Date", "pred"]].to_string(index=False))

sub = pd.DataFrame({
    "Id": smoke["Store"].astype(str) + "_" + smoke["Dept"].astype(str) + "_" +
          smoke["Date"].dt.strftime("%Y-%m-%d"),
    "Weekly_Sales": smoke["pred"]})
sub.to_csv("submission_xgboost.csv", index=False)

with mlflow.start_run(run_name="XGBoost_Submission"):
    mlflow.set_tags({"stage": "verification", "model_family": "XGBoost", "author": "adzid23"})
    mlflow.log_artifact("submission_xgboost.csv")
print("submission_xgboost.csv written:", sub.shape)

/tmp/ipykernel_491/1761351721.py:25: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df["is_target"] = df["is_target"].fillna(False).astype(bool)


rows: 115064 | NaNs: 0
      Date         pred
2012-11-30 25776.826172
2012-12-07 34586.148438
2012-12-14 43776.230469
2012-12-21 48216.113281
2012-12-28 26062.197266
🏃 View run XGBoost_Submission at: https://dagshub.com/adzid23/Freeuni_ML_Walmart_Sales_Forecasting.mlflow/#/experiments/9/runs/08f2200a8bd0402aba2f29bfc8aff446
🧪 View experiment at: https://dagshub.com/adzid23/Freeuni_ML_Walmart_Sales_Forecasting.mlflow/#/experiments/9
submission_xgboost.csv written: (115064, 2)


In [17]:
!kaggle competitions submit -c walmart-recruiting-store-sales-forecasting \
    -f submission_xgboost.csv -m "XGBoost 3-seed ensemble, v3 features, log1p"

100% 2.89M/2.89M [00:00<00:00, 4.14MB/s]
Successfully submitted to Walmart Recruiting - Store Sales Forecasting

In [18]:
!kaggle competitions submissions -c walmart-recruiting-store-sales-forecasting | head -5

fileName                 date                        description                                   status                     publicScore  privateScore  
-----------------------  --------------------------  --------------------------------------------  -------------------------  -----------  ------------  
submission_xgboost.csv   2026-07-12 17:43:27.667000  XGBoost 3-seed ensemble, v3 features, log1p   SubmissionStatus.COMPLETE  2707.44741   2796.79012    
submission_lightgbm.csv  2026-07-12 15:26:15.540000  LightGBM 3-seed ensemble, v3 features, log1p  SubmissionStatus.COMPLETE  2640.28322   2725.04534    
